# ATLAS Solar Satellite Preprocessing Tutorial

This notebook preprocesses monthly CERES satellite solar radiation data and exports a country-level NetCDF file.

It is designed as a step-by-step tutorial for users who are not Python experts. Users should only edit the **Input parameters** cell, then run the notebook cells in order.

The input is the global satellite download produced by the satellite download notebook. The output is saved using the structure:

`../data/processed/{variable}/{country}/`

For this workflow, the variable name is `ghi`.


## Step 1. Install and import required libraries

Run this cell first. If any package is missing, install it in your Python environment before continuing.


In [1]:
from pathlib import Path
from datetime import datetime
import glob
import os

from pyhdf.SD import SD, SDC
import geopandas as gpd
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray  # required for CRS handling through the .rio accessor


## Step 2. Input parameters

Edit only this cell for a new country or data folder.

Parameters:

- `country`: country name to extract from the global dataset. It must match the `NAME_EN` field in the country shapefile.
- `variable`: output variable name. For solar Global Horizontal Irradiance, keep `ghi`.
- `product`: product folder name. For the satellite workflow, keep `ceres`.
- `download_root`: root folder containing the satellite downloads.
- `download_path`: input folder generated by the satellite download notebook.
- `output_path`: folder where the processed country-level NetCDF will be written.
- `shapefile_path`: country boundary shapefile used to crop the global dataset.
- `start_date` and `end_date`: temporal range to process.


In [2]:
# Country to process. Example: "Ecuador", "Peru", "Argentina".
country = "Argentina"

# Product and variable settings.
product = "ceres"
variable = "ghi"

# Input path from the satellite download notebook.
# Expected structure: ../data/ceres_downloads/ghi/
download_root = Path("../data/ceres_downloads")
download_path = download_root / variable

# Output path for the processed country-level file.
# Expected structure: ../data/processed/ghi/ecuador/
country_slug = country.lower().replace(" ", "_")
output_path = Path("../data/processed") / variable / country_slug
output_path.mkdir(parents=True, exist_ok=True)

# Country boundaries used for spatial subsetting.
shapefile_path = Path("../world_map/ne_50m_admin_0_countries.shp")

# Period to process.
start_date = "2000-01"
end_date = "2026-03"

# CERES HDF variable corresponding to Global Horizontal Irradiance.
hdf_variable_name = "init_all_sfc_sw_dn_reg"

# Output filename.
output_filename = output_path / f"{variable}_{start_date}_{end_date}_processed.nc"


## Step 3. Support functions

These functions prepare coordinates, read country boundaries, load monthly CERES HDF files, crop the dataset and save the final NetCDF file.

Users normally do not need to edit this section.


In [3]:
def month_range(start: str, end: str) -> pd.DatetimeIndex:
    """Return a monthly date range from start YYYY-MM to end YYYY-MM, inclusive."""
    return pd.date_range(pd.to_datetime(start), pd.to_datetime(end), freq="MS")


def ensure_xarray_epsg4326(xdf: xr.Dataset) -> xr.Dataset:
    """Assign EPSG:4326 CRS and spatial dimensions to an xarray Dataset."""
    xdf = xdf.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude", inplace=False)
    if xdf.rio.crs is None:
        xdf = xdf.rio.write_crs("EPSG:4326", inplace=False)
    return xdf


def roll_longitudes(ds: xr.Dataset, lon_name: str = "longitude") -> xr.Dataset:
    """Convert longitudes from 0–360 degrees east to -180–180 degrees."""
    ds = ds.assign_coords({lon_name: ((ds[lon_name] + 180) % 360) - 180})
    return ds.sortby(lon_name)


def fix_coords(xdf: xr.Dataset) -> xr.Dataset:
    """Round coordinates, assign CRS and convert longitudes when needed."""
    xdf = ensure_xarray_epsg4326(xdf)
    xdf["longitude"] = np.round(xdf.longitude, 3)
    xdf["latitude"] = np.round(xdf.latitude, 3)

    if float(xdf.longitude.min()) >= 0:
        xdf = roll_longitudes(xdf)

    return xdf.sortby("latitude")


def get_country_geometry(country_name, shapefile):
    """Read the selected country geometry from the Natural Earth shapefile.

    This function is compatible with both recent and older GeoPandas versions.
    New GeoPandas versions provide ``union_all()``; older versions use
    ``unary_union``. The fallback avoids version-specific errors.
    """
    gdf = gpd.read_file(shapefile)

    if gdf.crs is None:
        gdf = gdf.set_crs(epsg=4326)
    else:
        gdf = gdf.to_crs(epsg=4326)

    matches = gdf[gdf["NAME_EN"].astype(str).str.lower() == country_name.lower()]

    if matches.empty:
        available_examples = ", ".join(sorted(gdf["NAME_EN"].dropna().unique())[:10])
        raise ValueError(
            f"Country '{country_name}' was not found in {shapefile}. "
            f"Check the English country name. Examples in the file include: {available_examples}"
        )

    if hasattr(matches.geometry, "union_all"):
        country_geometry = matches.geometry.union_all()
    else:
        country_geometry = matches.geometry.unary_union

    return gpd.GeoSeries([country_geometry], crs="EPSG:4326")
def find_ceres_monthly_file(folder: Path, year: int, month: int) -> Path:
    """Find the CERES monthly HDF file for a specific year and month."""
    ym = f"{year}{month:02d}"
    pattern = str(folder / f"CER_SYN1deg-Month_*.{ym}")
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No CERES file found for {year}-{month:02d} in {folder}")
    return Path(files[0])


def read_ceres_monthly_file(file_path: Path, hdf_variable: str, variable_name: str) -> xr.Dataset:
    """Read one CERES HDF monthly file and convert it to an xarray Dataset."""
    hdf = SD(str(file_path), SDC.READ)

    data = hdf.select(hdf_variable)[:]
    lat = hdf.select("latitude")[:]
    lon = hdf.select("longitude")[:]

    year_month = file_path.name.split(".")[-1]
    time = pd.to_datetime(f"{year_month[:4]}-{year_month[4:6]}")

    ds = xr.Dataset(
        {variable_name: (("latitude", "longitude"), data)},
        coords={"latitude": lat, "longitude": lon},
    )
    ds = ds.expand_dims(time=[time])
    return fix_coords(ds)


def load_ceres_dataset(folder: Path, hdf_variable: str, variable_name: str, start: str, end: str) -> xr.Dataset:
    """Load and concatenate all monthly CERES files in the selected period."""
    if not folder.exists():
        raise FileNotFoundError(f"Input download folder not found: {folder}")

    datasets = []
    missing_months = []

    for date in month_range(start, end):
        try:
            file_path = find_ceres_monthly_file(folder, date.year, date.month)
            datasets.append(read_ceres_monthly_file(file_path, hdf_variable, variable_name))
        except FileNotFoundError:
            missing_months.append(date.strftime("%Y-%m"))

    if not datasets:
        raise FileNotFoundError(
            f"No CERES files were found in {folder} for the selected period {start} to {end}."
        )

    if missing_months:
        print(f"Warning: {len(missing_months)} monthly files were not found and will be skipped.")
        print("Missing months:", ", ".join(missing_months))

    return xr.concat(datasets, dim="time").sortby("time")


def crop_to_geometry(xdf: xr.Dataset, geometries: gpd.GeoSeries, buffer_deg: float = 2.0) -> xr.Dataset:
    """Crop a global xarray Dataset to a country bounding box with optional buffer."""
    xdf = xdf.sortby("latitude")
    lon_min, lat_min, lon_max, lat_max = geometries.total_bounds

    return xdf.sel(
        longitude=slice(lon_min - buffer_deg, lon_max + buffer_deg),
        latitude=slice(lat_min - buffer_deg, lat_max + buffer_deg),
    )


def save_netcdf(xdf: xr.Dataset, output_file: Path, overwrite: bool = True) -> None:
    """Save an xarray Dataset as NetCDF using float32 encoding."""
    output_file.parent.mkdir(parents=True, exist_ok=True)

    if output_file.exists() and not overwrite:
        print(f"File already exists and overwrite is False: {output_file}")
        return

    xdf = xdf.chunk({"time": 50, "latitude": 256, "longitude": 256})
    encoding = {var: {"dtype": "float32"} for var in xdf.data_vars}

    xdf.to_netcdf(output_file, engine="netcdf4", encoding=encoding)
    print(f"Processed NetCDF written to: {output_file}")


### Compatibility note

The country geometry function has been updated to work with both older and newer GeoPandas versions.  
If you see a message such as `PROJ: proj_create_from_database`, it usually means the local conda environment is pointing to a missing PROJ data folder. The notebook now avoids the GeoPandas `union_all()` compatibility issue, which was the actual cause of the traceback.

## Step 4. Run preprocessing

Run this cell to load the global CERES satellite files, crop them to the selected country and save the final processed NetCDF file.


In [4]:
# 1. Read the country boundary.
geometries = get_country_geometry(country, shapefile_path)

# 2. Load the global satellite download for the selected variable and period.
global_satellite_dataset = load_ceres_dataset(
    folder=download_path,
    hdf_variable=hdf_variable_name,
    variable_name=variable,
    start=start_date,
    end=end_date,
)

# 3. Crop the global dataset to the selected country.
country_dataset = crop_to_geometry(global_satellite_dataset, geometries, buffer_deg=2.0)

# 4. Save the processed country-level dataset.
save_netcdf(country_dataset, output_filename, overwrite=True)


ERROR 1: PROJ: proj_create_from_database: Open of /home/alessandrom/anaconda3/envs/bias_correction_conda/share/proj failed


Missing months: 2000-01, 2000-02, 2001-04, 2001-05, 2001-06, 2001-07, 2001-08, 2001-09, 2001-10, 2001-11, 2001-12, 2002-01, 2002-02, 2002-03, 2002-04, 2002-05, 2002-06, 2002-07, 2002-08, 2002-09, 2002-10, 2002-11, 2002-12, 2003-01, 2003-02, 2003-03, 2003-04, 2003-05, 2003-06, 2003-07, 2003-08, 2003-09, 2003-10, 2003-11, 2003-12, 2004-01, 2004-02, 2004-03, 2004-04, 2004-05, 2004-06, 2004-07, 2004-08, 2004-09, 2004-10, 2004-11, 2004-12, 2005-01, 2005-02, 2005-03, 2005-04, 2005-05, 2005-06, 2005-07, 2005-08, 2005-09, 2005-10, 2005-11, 2005-12, 2006-01, 2006-02, 2006-03, 2006-04, 2006-05, 2006-06, 2006-07, 2006-08, 2006-09, 2006-10, 2006-11, 2006-12, 2007-01, 2007-02, 2007-03, 2007-04, 2007-05, 2007-06, 2007-07, 2007-08, 2007-09, 2007-10, 2007-11, 2007-12, 2008-01, 2008-02, 2008-03, 2008-04, 2008-05, 2008-06, 2008-07, 2008-08, 2008-09, 2008-10, 2008-11, 2008-12, 2009-01, 2009-02, 2009-03, 2009-04, 2009-05, 2009-06, 2009-07, 2009-08, 2009-09, 2009-10, 2009-11, 2009-12, 2010-01, 2010-02, 201

## Step 5. Check the output

This optional cell opens the processed file and shows a compact summary to confirm that the result was created successfully.


In [5]:
processed_dataset = xr.open_dataset(output_filename)
processed_dataset


/home/alessandrom/anaconda3/envs/bias_correction_conda/lib/python3.10/site-packages/gribapi/__init__.py:23: UserWarning: ecCodes 2.31.0 or higher is recommended. You are running version 2.16.0
  warnings.warn(


<xarray.Dataset> Size: 47kB
Dimensions:      (time: 13, latitude: 37, longitude: 24)
Coordinates:
  * time         (time) datetime64[ns] 104B 2000-03-01 2000-04-01 ... 2001-03-01
  * latitude     (latitude) float32 148B -56.5 -55.5 -54.5 ... -22.5 -21.5 -20.5
  * longitude    (longitude) float32 96B -75.5 -74.5 -73.5 ... -54.5 -53.5 -52.5
Data variables:
    ghi          (time, latitude, longitude) float32 46kB ...
    spatial_ref  int64 8B ...